# 3-3. 단어 임베딩

> **이 노트북에서 답할 질문**
> 단어를 벡터로 바꾼다는 게 정확히 무슨 연산이고, 그게 어떻게 희소성 문제를 푸는가?

## 1. 원-핫 벡터: 무엇이 문제인가

단어를 숫자로 바꾸는 가장 단순한 방법은 **원-핫 인코딩**입니다.
어휘가 V개면 길이 V짜리 벡터를 만들고, 해당 단어 자리만 1로 둡니다.

```
기계  -> [0, 0, 1, 0, 0, ..., 0]
장치  -> [0, 0, 0, 0, 1, ..., 0]
고양이 -> [1, 0, 0, 0, 0, ..., 0]
```

문제가 보이시나요? **세 벡터가 서로 완전히 직교합니다.**
기계-장치의 유사도와 기계-고양이의 유사도가 **똑같이 0**입니다.

이것이 3-1에서 본 저주의 수학적 정체입니다. 단어 사이에 관계를 표현할 자리가 없습니다.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from common import load_sentences, set_seed, get_device, count_params, human

set_seed(1234)
device = get_device()

In [ ]:
sents = load_sentences()

def tokenize(s):
    return s.replace(".", " .").split()

vocab = sorted({t for s in sents for t in tokenize(s)})
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
V = len(vocab)
print(f"어휘 크기 V = {V}")

def onehot(word):
    v = torch.zeros(V)
    v[stoi[word]] = 1.0
    return v

a, b, c = "모델은", "모델", "고양이" if "고양이" in stoi else vocab[0]
print(f"\n원-핫 벡터 차원: {onehot(a).shape}")
print(f"cos(기계류 단어끼리)  : {F.cosine_similarity(onehot('언어'), onehot('모델은'), dim=0):.3f}")
print(f"cos(전혀 다른 단어끼리): {F.cosine_similarity(onehot('언어'), onehot('혼란도가'), dim=0):.3f}")
print("\n-> 모든 쌍이 0. 원-핫은 '다르다'만 말할 뿐 '얼마나 다른지'를 못 말합니다.")

## 2. 임베딩 = 조회표(lookup table)

해결책은 단순합니다. 단어마다 **짧은 실수 벡터**를 하나씩 붙여 줍니다.

```
          차원 0   차원 1   차원 2  ...  차원 d-1
기계      0.31    -0.82    0.05   ...   0.44
장치      0.28    -0.79    0.11   ...   0.39     <- 기계와 비슷한 위치
고양이   -0.91     0.22    0.67   ...  -0.13     <- 멀리 떨어짐
```

이 표가 바로 **임베딩 행렬** $E \in \mathbb{R}^{V \times d}$ 입니다.
중요한 건 **이 숫자들을 사람이 정하지 않는다**는 것. 학습으로 얻습니다.

### 수학적으로는 그냥 행렬 곱

원-핫 벡터에 $E$ 를 곱하면 해당 행이 그대로 튀어나옵니다.

$$\text{onehot}(w)^\top E = E[w]$$

그런데 0이 대부분인 벡터를 곱하는 건 낭비죠. 그래서 파이토치는
`nn.Embedding` 으로 **곱셈 없이 인덱싱만** 합니다. 결과는 동일합니다.

In [ ]:
d = 8   # 임베딩 차원
emb = nn.Embedding(V, d)

w = "모델은"
idx = torch.tensor([stoi[w]])

# (a) nn.Embedding 으로 조회
via_lookup = emb(idx)

# (b) 원-핫 x 행렬곱 으로 계산
via_matmul = onehot(w).unsqueeze(0) @ emb.weight

print(f"조회 결과   : {via_lookup.detach().numpy().round(4)}")
print(f"행렬곱 결과 : {via_matmul.detach().numpy().round(4)}")
print(f"\n같은가? {torch.allclose(via_lookup, via_matmul, atol=1e-6)}")
print(f"\n임베딩 행렬 크기: {tuple(emb.weight.shape)}  = V x d")
print(f"파라미터 수: {human(count_params(emb))}")

### 차원 수가 만드는 차이

원-핫은 V차원, 임베딩은 d차원입니다. 보통 $d \ll V$ 입니다.

In [ ]:
print(f"{'어휘 V':>10} | {'원-핫 차원':>10} | {'임베딩 d=256':>13} | {'압축률':>8}")
print("-" * 50)
for V_ex in (1_000, 32_000, 128_000):
    print(f"{V_ex:>10,} | {V_ex:>10,} | {256:>13,} | {V_ex/256:>7.0f}x")
print()
print("차원이 줄어든 게 핵심이 아닙니다.")
print("핵심은 d개의 축을 '공유'하면서 단어들이 서로 관계를 맺을 수 있게 된 것입니다.")

## 3. 임베딩은 어떻게 학습되는가

임베딩 값을 정하는 원리는 **분포 가설(distributional hypothesis)** 입니다.

> "비슷한 문맥에 나타나는 단어는 비슷한 의미다."

그래서 "다음 단어 맞히기"를 시키면 임베딩이 저절로 정렬됩니다.
`언어 모델은 ___` 뒤에 올 단어를 맞히는 훈련을 반복하면,
그 자리에 올 수 있는 단어들의 벡터가 점점 가까워집니다.

작은 모델로 직접 확인해 봅시다. **앞 2개 단어로 다음 단어 맞히기**입니다.

In [ ]:
# 학습 데이터: (앞 2단어, 다음 1단어) 쌈
CTX = 2
pairs = []
for s in sents:
    toks = ["<s>"] * CTX + tokenize(s)
    for i in range(CTX, len(toks)):
        if all(t in stoi or t == "<s>" for t in toks[i-CTX:i+1]):
            pairs.append(([toks[j] for j in range(i-CTX, i)], toks[i]))

# <s> 를 어휘에 추가
if "<s>" not in stoi:
    stoi["<s>"] = V
    itos[V] = "<s>"
    vocab = vocab + ["<s>"]
    V = len(vocab)

X = torch.tensor([[stoi[w] for w in ctx] for ctx, _ in pairs])
Y = torch.tensor([stoi[w] for _, w in pairs])
print(f"학습 샘플 {len(X)}개,  X.shape={tuple(X.shape)},  V={V}")
print(f"예시: {[itos[i.item()] for i in X[0]]} -> {itos[Y[0].item()]}")

In [ ]:
class TinyLM(nn.Module):
    """임베딩 -> 이어붙이기 -> 은닉층 -> 다음 단어 확률."""
    def __init__(self, V, d=16, hidden=64, ctx=CTX):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.net = nn.Sequential(
            nn.Linear(ctx * d, hidden), nn.Tanh(), nn.Linear(hidden, V)
        )

    def forward(self, x):                 # x: (B, ctx)
        e = self.emb(x)                   # (B, ctx, d)
        return self.net(e.flatten(1))     # (B, V)

model = TinyLM(V).to(device)
print(f"파라미터 수: {human(count_params(model))}")
print(f"학습 전 손실 기대값 log(V) = {math.log(V):.3f},  혼란도 = {V}")

In [ ]:
Xd, Yd = X.to(device), Y.to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)

for step in range(1, 601):
    logits = model(Xd)
    loss = F.cross_entropy(logits, Yd)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0:
        print(f"step {step:4d}  loss {loss.item():.4f}  ppl {math.exp(loss.item()):7.2f}")

손실이 $\log V$ 에서 확 떨어졌습니다. (이 말뭉치가 작아서 사실상 외운 것에 가깝습니다 —
3-2에서 본 과적합입니다. 지금은 **임베딩이 정렬되는지**만 보는 게 목적입니다.)

## 4. 학습된 임베딩 들여다보기

이제 임베딩 공간에서 어떤 단어들이 가까워졌는지 봅시다.
비교는 **코사인 유사도**로 합니다. 벡터의 길이가 아니라 **방향**만 보는 척도입니다.

$$\cos(u, v) = \frac{u \cdot v}{\|u\| \|v\|}$$

In [ ]:
E = model.emb.weight.detach().cpu()
En = F.normalize(E, dim=1)          # 길이를 1로 맞추면 내적 = 코사인 유사도

def nearest(word, k=5):
    if word not in stoi:
        return f"'{word}' 는 어휘에 없습니다"
    sims = En @ En[stoi[word]]
    top = sims.topk(k + 1).indices.tolist()
    return [(itos[i], round(sims[i].item(), 3)) for i in top if i != stoi[word]][:k]

for w in ["모델은", "학습은", "확률을", "혼란도는"]:
    if w in stoi:
        print(f"{w:>8} 와 가까운 단어: {nearest(w)}")

> 말뭉치가 50문장뿐이라 결과가 깔끔하진 않습니다. **크기가 전부인 영역**이라는 걸
> 체감하는 것도 공부입니다. 연습 1번에서 큰 말뭉치로 다시 해 보세요.

## 5. 왜 이게 희소성을 푸는가 — 핵심 장면

이제 3-1의 질문으로 돌아갑시다.

> `언어 모델은 ___` 에서 `기계이다`는 봤고 `장치이다`는 못 봤다. N-gram은 둘을 구분 못 한다.

임베딩 모델은 다릅니다. **한 번도 본 적 없는 문맥**을 넣어 봐도 그럴듯한 답을 내놓습니다.
문맥 단어들의 벡터가 비슷하면 은닉층 출력도 비슷하고, 따라서 예측도 비슷해지기 때문입니다.

In [ ]:
@torch.no_grad()
def predict(ctx_words, k=5):
    ids = torch.tensor([[stoi.get(w, stoi['<s>']) for w in ctx_words]]).to(device)
    p = F.softmax(model(ids)[0], dim=-1).cpu()
    top = p.topk(k).indices.tolist()
    return [(itos[i], round(p[i].item(), 3)) for i in top]

seen_contexts = set(tuple(ctx) for ctx, _ in pairs)

tests = [["언어", "모델은"], ["뉴럴", "모델은"], ["통계적", "모델은"]]
for t in tests:
    mark = "학습에서 봄" if tuple(t) in seen_contexts else "처음 보는 문맥"
    print(f"{' '.join(t):>14} ({mark:>12}) -> {predict(t, 3)}")

**처음 보는 문맥에도 0이 아닌, 심지어 그럴듯한 확률이 나옵니다.**

N-gram이라면 여기서 그냥 손을 들었을 자리입니다.
임베딩이 만든 **일반화(generalization)** 가 바로 이것입니다.

### 한 문장 요약

> N-gram은 **조합을 외웠고**, 뉴럴 언어 모델은 **공간을 배웠다.**
> 공간 위에서는 가 본 적 없는 좌표도 보간할 수 있다.

## 6. 임베딩 차원 d는 어떻게 정하나

In [ ]:
STEPS = 100   # 일부러 짧게 -- '얼마나 빨리 배우는가'를 봅니다
for d_try in (2, 8, 32, 128):
    set_seed(1234)
    m = TinyLM(V, d=d_try).to(device)
    o = torch.optim.Adam(m.parameters(), lr=0.01)
    for _ in range(STEPS):
        l = F.cross_entropy(m(Xd), Yd)
        o.zero_grad(); l.backward(); o.step()
    print(f"d={d_try:>4}  파라미터 {human(count_params(m)):>8}  {STEPS}스텝 후 loss {l.item():.4f}  ppl {math.exp(l.item()):7.2f}")

### 결과 해석 — 여기서 두 가지를 배웁니다

**1) d가 너무 작으면 확실히 나쁩니다.** `d=2`는 285개 단어를 평면 위에 욱여넣는 셈이라
서로를 구분할 자리가 없습니다. 눈에 띄게 뒤처집니다.

**2) 어느 선을 넘으면 더 키워도 소용이 없습니다.** d=8, 32, 128의 차이가 거의 없죠.
이 말뭉치가 담고 있는 정보량이 d=8로도 충분히 표현되기 때문입니다.
**모델 크기는 데이터가 허락하는 만큼만 의미가 있습니다** — LLM 스케일링의 축소판입니다.

그리고 주의: 여기서 잰 건 **학습 손실**입니다. d를 무한정 키우면 학습 손실은 계속 떨어지지만
평가 손실은 어느 지점부터 다시 올라갑니다(과적합). 진짜 d를 고를 땐 반드시 평가 손실로 재세요.

실무 감각:

| 모델 규모 | 임베딩 차원 d |
|---|---|
| 장난감/실습 | 32 ~ 128 |
| 작은 GPT | 384 ~ 768 |
| 대형 LLM | 4096 ~ 8192 |

---

## 정리

- **원-핫**: 모든 단어가 직교 → 관계 표현 불가 → N-gram 저주의 정체
- **임베딩**: $V \times d$ 조회표. 원-핫 행렬곱과 수학적으로 동일하지만 인덱싱으로 구현
- **학습 원리**: 분포 가설. 다음 단어 맞히기만 시켜도 비슷한 단어가 모임
- **효과**: 처음 보는 문맥에도 그럴듯한 확률 → **일반화**

## 연습

1. 위키백과 한국어 문서를 긁어 텍스트 파일로 만들고 `load_sentences(path)` 로 넘겨
   `nearest()` 결과가 얼마나 좋아지는지 보세요. **이 차이가 데이터 규모의 힘입니다.**
2. `d=2` 로 학습한 뒤 임베딩을 matplotlib 산점도로 그려 보세요. 눈으로 군집이 보이나요?
3. `TinyLM` 에서 `nn.Tanh()` 를 `nn.ReLU()` 로 바꾸면 수렴이 달라지나요?

## 다음

→ `../Ch4/0401_NPLM.ipynb` : 이 TinyLM을 제대로 된 **NPLM(Bengio 2003)** 으로 키우기